SECOND SUBMISSION : Data Preparation and Baseline

Import different datasets 

In [16]:
from pathlib import Path
import pandas as pd

HERE = Path(".")

df_roads = pd.read_csv(
    HERE / "denominations-emprises-voies-actuelles.csv", sep=";", low_memory=False
)
df_water = pd.read_csv(
    HERE / "plan-de-voirie-voies-deau.csv", sep=";", low_memory=False
)
df_green_spaces = pd.read_csv(
    HERE / "espaces_verts.csv", sep=";", low_memory=False
)
df_weather_prev = pd.read_csv(
    HERE / "Q_75_previous-1950-2024_RR-T-Vent.csv.gz", sep=";", low_memory=False
)
df_weather_latest = pd.read_csv(
    HERE / "Q_75_latest-2025-2026_RR-T-Vent.csv.gz", sep=";", low_memory=False
)
df_tmy_raw = pd.read_csv(
    HERE / "tmy_48.859_2.320_2005_2023.csv", nrows=8760
)

Cleaning weather data 

In [17]:
SUMMER_MONTHS = [5, 6, 7, 8, 9]

weather_rename_map = {
    "RR": "precipitation_mm_24h",
    "TN": "temp_min_c",
    "HTN": "hour_temp_min",
    "TX": "temp_max_c",
    "HTX": "hour_temp_max",
    "TM": "temp_mean_c",
    "TNTXM": "temp_mean_min_max_c",
    "TAMPLI": "temp_amplitude_c",
    "DG": "frost_duration_min",
    "FFM": "wind_mean_ms",
}

df_weather = pd.concat([df_weather_prev, df_weather_latest], ignore_index=True)
df_weather["DATE"] = pd.to_datetime(
    df_weather["AAAAMMJJ"].astype(str), format="%Y%m%d", errors="coerce"
)

# Filtrage : >= 1990 et mois de mai à septembre
df_weather = df_weather[
    (df_weather["DATE"].dt.year >= 1990)
    & (df_weather["DATE"].dt.month.isin(SUMMER_MONTHS))
].copy()

# Suppression des codes qualité Q* et des colonnes inutiles / > 80% manquants
q_cols = [c for c in df_weather.columns if c.startswith("Q") and len(c) > 1]
high_na_cols = df_weather.columns[df_weather.isna().mean() > 0.80].tolist()
extra_drop = ["STATUS_FXI3S", "STATUS_DXI3S", "AAAAMMJJ"]

# Si tu veux conserver le vent moyen (FFM), retire "FFM" de la liste supprimée :
cols_to_drop = list(set(q_cols + high_na_cols + extra_drop) - {"FFM"})

df_weather_clean = (
    df_weather
    .drop(columns=cols_to_drop, errors="ignore")
    .rename(columns={k: v for k, v in weather_rename_map.items() if k in df_weather.columns})
    .copy()
)

# Extraction de l'arrondissement (1 à 20)
df_weather_clean["arrondissement"] = (
    pd.to_numeric(df_weather_clean["NUM_POSTE"].astype(str).str[:5], errors="coerce") - 75100
).astype("Int64")
df_weather_clean.loc[~df_weather_clean["arrondissement"].between(1, 20), "arrondissement"] = pd.NA

print("Weather clean shape :", df_weather_clean.shape)
print("Weather columns     :", list(df_weather_clean.columns))

Weather clean shape : (82538, 17)
Weather columns     : ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'precipitation_mm_24h', 'temp_min_c', 'hour_temp_min', 'temp_max_c', 'hour_temp_max', 'temp_mean_c', 'temp_mean_min_max_c', 'temp_amplitude_c', 'frost_duration_min', 'wind_mean_ms', 'DATE', 'arrondissement']


Profil journalier TMY (par mois, jour)

In [18]:
tmy_dt = pd.to_datetime(df_tmy_raw["time(UTC)"], format="%Y%m%d:%H%M", errors="coerce")
tmy_cols = ["Gb(n)", "WS10m", "WD10m", "SP"]

df_tmy_daily = (
    df_tmy_raw[tmy_cols]
    .apply(pd.to_numeric, errors="coerce")
    .assign(
        month=tmy_dt.dt.month,
        day=tmy_dt.dt.day,
    )
    .dropna(subset=["month", "day"])
    .groupby(["month", "day"], as_index=False)
    .mean()
    .rename(columns={
        "Gb(n)": "dni_wm2",
        "WS10m": "wind_speed_10m_ms",
        "WD10m": "wind_dir_10m_deg",
        "SP": "pressure_pa",
    })
)

print("TMY daily shape :", df_tmy_daily.shape)
df_tmy_daily.head()

TMY daily shape : (365, 6)


,month,day,dni_wm2,wind_speed_10m_ms,wind_dir_10m_deg,pressure_pa
0,1,1,54.334583,4.183333,266.833333,99546.666667
1,1,2,164.162500,2.747083,246.125000,101337.083333
2,1,3,64.489167,1.386250,233.291667,102127.083333
3,1,4,99.993333,2.511250,99.166667,101823.750000
4,1,5,8.931250,2.673333,43.250000,100976.250000


Agrégations par arrondissement (espaces verts, routes, eau)

In [ ]:
# 1. Espaces verts
df_green_spaces["arrondissement"] = (
    pd.to_numeric(df_green_spaces["Code postal"], errors="coerce") - 75000
).astype("Int64")
df_green_spaces.loc[~df_green_spaces["arrondissement"].between(1, 20), "arrondissement"] = pd.NA

for col in ["Surface calculée", "Superficie totale réelle", "Surface horticole", "Périmètre"]:
    if col in df_green_spaces.columns:
        df_green_spaces[col] = pd.to_numeric(
            df_green_spaces[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

df_green_agg = (
    df_green_spaces.dropna(subset=["arrondissement"])
    .groupby("arrondissement", as_index=False)
    .agg(
        green_count=("Identifiant espace vert", "count"),
        green_total_area=("Superficie totale réelle", "sum"),
        green_calc_area=("Surface calculée", "sum"),
        green_hort_area=("Surface horticole", "sum"),
        green_perimeter=("Périmètre", "sum"),
        main_green_category=(
            "Catégorie",
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA,
        ),
    )
)

arr_area_ha = {
    1: 183, 2: 99, 3: 117, 4: 160, 5: 254,
    6: 215, 7: 409, 8: 388, 9: 218, 10: 289,
    11: 367, 12: 637, 13: 715, 14: 564, 15: 848,
    16: 791, 17: 567, 18: 601, 19: 679, 20: 598,
}
total_green_paris = df_green_agg["green_total_area"].sum()
df_green_agg["green_area_share_pct"] = ((df_green_agg["green_total_area"] / total_green_paris) * 100).round(2)
df_green_agg["arr_area_m2"] = df_green_agg["arrondissement"].map(arr_area_ha) * 10_000
df_green_agg["green_area_pct"] = ((df_green_agg["green_total_area"] / df_green_agg["arr_area_m2"]) * 100).round(2)

# 2. Voirie (format large par type de voie)
df_roads["arrondissement"] = pd.to_numeric(
    df_roads["Arrondissement"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce",
).astype("Int64")
df_roads.loc[~df_roads["arrondissement"].between(1, 20), "arrondissement"] = pd.NA
df_roads["road_type"] = df_roads["Type voie"].astype(str).str.strip().str.lower()

for col in ["Longueur voie", "Largeur voie"]:
    df_roads[col] = pd.to_numeric(
        df_roads[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

df_roads["road_surface_m2"] = df_roads["Longueur voie"] * df_roads["Largeur voie"]
target_types = ["rue", "boulevard", "avenue"]

df_roads_agg = (
    df_roads[df_roads["arrondissement"].notna() & df_roads["road_type"].isin(target_types)]
    .groupby(["arrondissement", "road_type"], as_index=False)
    .agg(
        roads_total_length_m=("Longueur voie", "sum"),
        roads_avg_width_m=("Largeur voie", "mean"),
        roads_total_surface_m2=("road_surface_m2", "sum"),
    )
)

df_roads_pivot = df_roads_agg.pivot(
    index="arrondissement",
    columns="road_type",
    values=["roads_total_length_m", "roads_avg_width_m", "roads_total_surface_m2"],
)
df_roads_pivot.columns = [f"{metric}_{rt}" for metric, rt in df_roads_pivot.columns]
df_roads_pivot = df_roads_pivot.reset_index()

# 3. Eau (rattachement au centroïde d'arrondissement le plus proche)
df_water_clean = df_water.copy()
for col in ["st_area(shape)", "st_perimeter(shape)"]:
    if col in df_water_clean.columns:
        df_water_clean[col] = pd.to_numeric(
            df_water_clean[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

coords = df_water_clean["geo_point_2d"].astype(str).str.split(",", expand=True)
df_water_clean["water_lat"] = pd.to_numeric(coords[0], errors="coerce")
df_water_clean["water_lon"] = pd.to_numeric(coords[1], errors="coerce")

green_coords = df_green_spaces["Geo point"].astype(str).str.split(",", expand=True)
arr_centroids = (
    df_green_spaces.assign(
        g_lat=pd.to_numeric(green_coords[0], errors="coerce"),
        g_lon=pd.to_numeric(green_coords[1], errors="coerce"),
    )
    .dropna(subset=["arrondissement", "g_lat", "g_lon"])
    .groupby("arrondissement", as_index=False)[["g_lat", "g_lon"]]
    .mean()
)

def nearest_arr(row):
    if pd.isna(row["water_lat"]) or pd.isna(row["water_lon"]):
        return pd.NA
    dist2 = (arr_centroids["g_lat"] - row["water_lat"]) ** 2 + (arr_centroids["g_lon"] - row["water_lon"]) ** 2
    return arr_centroids.loc[dist2.idxmin(), "arrondissement"]

df_water_clean["arrondissement"] = df_water_clean.apply(nearest_arr, axis=1).astype("Int64")

df_water_agg = (
    df_water_clean.dropna(subset=["arrondissement"])
    .groupby("arrondissement", as_index=False)
    .agg(
        water_count=("OBJECTID", "count"),
        water_total_area=("st_area(shape)", "sum"),
        main_water_class=(
            "LIB_CLASSE",
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA,
        ),
    )
)

New shape: (82538, 35)
Months kept: [5, 6, 7, 8, 9]
Remaining columns: ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMMJJ', 'RR', 'TN', 'HTN', 'TX', 'HTX', 'TM', 'TNTXM', 'TAMPLI', 'TNSOL', 'TN50', 'DG', 'FFM', 'FF2M', 'FXY', 'DXY', 'HXY', 'FXI', 'DXI', 'HXI', 'FXI2', 'DXI2', 'HXI2', 'FXI3S', 'DXI3S', 'HXI3S', 'DRR', 'STATUS_FXI3S', 'STATUS_DXI3S', 'DATE']

Missing values (%):
 HXI2            100.0
DXI2            100.0
FXI2            100.0
FF2M            100.0
DRR              95.7
TN50             93.2
TNSOL            91.2
DXY              85.1
HXY              85.0
FXY              85.0
HXI3S            84.4
DXI3S            83.1
STATUS_DXI3S     83.1
DXI              83.1
STATUS_FXI3S     83.0
HXI              83.0
FXI              83.0
FXI3S            83.0
FFM              82.7
DG               75.6
HTN              73.9
HTX              73.9
TM               73.5
TAMPLI           46.4
TNTXM            46.4
TX               46.1
TN               46.1
RR               

In [7]:
high_missing_cols = df_weather_clean.columns[df_weather_clean.isna().mean() > 0.80].tolist()
extra_drop = [c for c in ["STATUS_FXI3S", "STATUS_DXI3S", "AAAAMMJJ"] if c in df_weather_clean.columns]

df_weather_clean = df_weather_clean.drop(columns=list(set(high_missing_cols + extra_drop))).copy()

print("Updated weather shape:", df_weather_clean.shape)
print("Remaining columns:", list(df_weather_clean.columns))

for col in ["TN", "TM", "TX", "RR", "FFM"]:
    if col in df_weather_clean.columns:
        print(f"{col:5s} -> min: {df_weather_clean[col].min()}, max: {df_weather_clean[col].max()}, missing: {df_weather_clean[col].isna().mean()*100:.1f}%")

Updated weather shape: (198464, 15)
Remaining columns: ['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'RR', 'TN', 'HTN', 'TX', 'HTX', 'TM', 'TNTXM', 'TAMPLI', 'DG', 'DATE']
TN    -> min: -13.8, max: 29.8, missing: 46.7%
TM    -> min: -11.8, max: 35.2, missing: 74.1%
TX    -> min: -9.3, max: 42.6, missing: 46.7%
RR    -> min: 0.0, max: 192.4, missing: 7.9%


extract the wanted 20 arrondissements

In [8]:
df_weather_clean["arrondissement"] = (
    pd.to_numeric(df_weather_clean["NUM_POSTE"].astype(str).str[:5], errors="coerce") - 75100
).astype("Int64")

valid_arr = df_weather_clean["arrondissement"].between(1, 20)
df_weather_clean.loc[~valid_arr, "arrondissement"] = pd.NA

print("Weather arrondissements:", sorted(df_weather_clean["arrondissement"].dropna().unique().tolist()))
print("Missing arrondissement in weather:", df_weather_clean["arrondissement"].isna().sum())

Weather arrondissements: [4, 6, 7, 10, 12, 13, 14, 15, 16, 17, 19, 20]
Missing arrondissement in weather: 0


clean green spaces dataset

In [20]:
# 1. Espaces verts
df_green_spaces["arrondissement"] = (
    pd.to_numeric(df_green_spaces["Code postal"], errors="coerce") - 75000
).astype("Int64")
df_green_spaces.loc[~df_green_spaces["arrondissement"].between(1, 20), "arrondissement"] = pd.NA

for col in ["Surface calculée", "Superficie totale réelle", "Surface horticole", "Périmètre"]:
    if col in df_green_spaces.columns:
        df_green_spaces[col] = pd.to_numeric(
            df_green_spaces[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

df_green_agg = (
    df_green_spaces.dropna(subset=["arrondissement"])
    .groupby("arrondissement", as_index=False)
    .agg(
        green_count=("Identifiant espace vert", "count"),
        green_total_area=("Superficie totale réelle", "sum"),
        green_calc_area=("Surface calculée", "sum"),
        green_hort_area=("Surface horticole", "sum"),
        green_perimeter=("Périmètre", "sum"),
        main_green_category=(
            "Catégorie",
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA,
        ),
    )
)

arr_area_ha = {
    1: 183, 2: 99, 3: 117, 4: 160, 5: 254,
    6: 215, 7: 409, 8: 388, 9: 218, 10: 289,
    11: 367, 12: 637, 13: 715, 14: 564, 15: 848,
    16: 791, 17: 567, 18: 601, 19: 679, 20: 598,
}
total_green_paris = df_green_agg["green_total_area"].sum()
df_green_agg["green_area_share_pct"] = ((df_green_agg["green_total_area"] / total_green_paris) * 100).round(2)
df_green_agg["arr_area_m2"] = df_green_agg["arrondissement"].map(arr_area_ha) * 10_000
df_green_agg["green_area_pct"] = ((df_green_agg["green_total_area"] / df_green_agg["arr_area_m2"]) * 100).round(2)

# 2. Voirie (format large par type de voie)
df_roads["arrondissement"] = pd.to_numeric(
    df_roads["Arrondissement"].astype(str).str.extract(r"(\d+)")[0],
    errors="coerce",
).astype("Int64")
df_roads.loc[~df_roads["arrondissement"].between(1, 20), "arrondissement"] = pd.NA
df_roads["road_type"] = df_roads["Type voie"].astype(str).str.strip().str.lower()

for col in ["Longueur voie", "Largeur voie"]:
    df_roads[col] = pd.to_numeric(
        df_roads[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

df_roads["road_surface_m2"] = df_roads["Longueur voie"] * df_roads["Largeur voie"]
target_types = ["rue", "boulevard", "avenue"]

df_roads_agg = (
    df_roads[df_roads["arrondissement"].notna() & df_roads["road_type"].isin(target_types)]
    .groupby(["arrondissement", "road_type"], as_index=False)
    .agg(
        roads_total_length_m=("Longueur voie", "sum"),
        roads_avg_width_m=("Largeur voie", "mean"),
        roads_total_surface_m2=("road_surface_m2", "sum"),
    )
)

df_roads_pivot = df_roads_agg.pivot(
    index="arrondissement",
    columns="road_type",
    values=["roads_total_length_m", "roads_avg_width_m", "roads_total_surface_m2"],
)
df_roads_pivot.columns = [f"{metric}_{rt}" for metric, rt in df_roads_pivot.columns]
df_roads_pivot = df_roads_pivot.reset_index()

# 3. Eau (rattachement au centroïde d'arrondissement le plus proche)
df_water_clean = df_water.copy()
for col in ["st_area(shape)", "st_perimeter(shape)"]:
    if col in df_water_clean.columns:
        df_water_clean[col] = pd.to_numeric(
            df_water_clean[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

coords = df_water_clean["geo_point_2d"].astype(str).str.split(",", expand=True)
df_water_clean["water_lat"] = pd.to_numeric(coords[0], errors="coerce")
df_water_clean["water_lon"] = pd.to_numeric(coords[1], errors="coerce")

green_coords = df_green_spaces["Geo point"].astype(str).str.split(",", expand=True)
arr_centroids = (
    df_green_spaces.assign(
        g_lat=pd.to_numeric(green_coords[0], errors="coerce"),
        g_lon=pd.to_numeric(green_coords[1], errors="coerce"),
    )
    .dropna(subset=["arrondissement", "g_lat", "g_lon"])
    .groupby("arrondissement", as_index=False)[["g_lat", "g_lon"]]
    .mean()
)

def nearest_arr(row):
    if pd.isna(row["water_lat"]) or pd.isna(row["water_lon"]):
        return pd.NA
    dist2 = (arr_centroids["g_lat"] - row["water_lat"]) ** 2 + (arr_centroids["g_lon"] - row["water_lon"]) ** 2
    return arr_centroids.loc[dist2.idxmin(), "arrondissement"]

df_water_clean["arrondissement"] = df_water_clean.apply(nearest_arr, axis=1).astype("Int64")

df_water_agg = (
    df_water_clean.dropna(subset=["arrondissement"])
    .groupby("arrondissement", as_index=False)
    .agg(
        water_count=("OBJECTID", "count"),
        water_total_area=("st_area(shape)", "sum"),
        main_water_class=(
            "LIB_CLASSE",
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA,
        ),
    )
)

merge all datasets & export 

In [21]:
# 1. Agrégation météo journalière par (arrondissement, DATE)
weather_num_cols = [
    c for c in df_weather_clean.select_dtypes(include="number").columns
    if c != "arrondissement"
]
weather_other_cols = [
    c for c in df_weather_clean.columns
    if c not in weather_num_cols + ["arrondissement", "DATE"]
]

agg_dict = {col: "mean" for col in weather_num_cols}
for col in weather_other_cols:
    agg_dict[col] = "first"

df_weather_base = (
    df_weather_clean.dropna(subset=["arrondissement", "DATE"])
    .groupby(["arrondissement", "DATE"], as_index=False)
    .agg(agg_dict)
)
df_weather_base["weather_source_arrondissement"] = df_weather_base["arrondissement"]
df_weather_base["weather_is_imputed"] = False

# 2. Imputation vers la station la plus proche pour les arrondissements sans station directe
nearest_weather_arr = {
    1: 4, 2: 10, 3: 4, 5: 4,
    8: 17, 9: 10, 11: 20, 18: 10,
}

imputed_parts = []
for target_arr, src_arr in nearest_weather_arr.items():
    sub = df_weather_base[df_weather_base["arrondissement"] == src_arr].copy()
    sub["arrondissement"] = target_arr
    sub["weather_source_arrondissement"] = src_arr
    sub["weather_is_imputed"] = True
    imputed_parts.append(sub)

df_weather_daily = pd.concat([df_weather_base] + imputed_parts, ignore_index=True)

# 3. Fusion globale + greffe TMY sur (month, day)
df_final = (
    df_green_agg
    .merge(df_weather_daily, on="arrondissement", how="left")
    .merge(df_roads_pivot, on="arrondissement", how="left")
    .merge(df_water_agg, on="arrondissement", how="left")
    .assign(
        month=lambda d: d["DATE"].dt.month,
        day=lambda d: d["DATE"].dt.day,
    )
    .merge(df_tmy_daily, on=["month", "day"], how="left")
    .drop(columns=["month", "day", "NUM_POSTE", "Code postal"], errors="ignore")
)

# 4. Diagnostic des valeurs manquantes et filtrage
missing_pct = (df_final.isna().mean() * 100).sort_values(ascending=False)
print("Top colonnes avec valeurs manquantes (%) :")
print(missing_pct.head(15).round(2))

essential_cols = [c for c in ["DATE", "temp_min_c", "temp_max_c"] if c in df_final.columns]
df_clean_essential = df_final.dropna(subset=essential_cols).copy()
df_all_complete = df_final.dropna().copy()

print("\nLignes avant filtre :", len(df_final))
print("Lignes avec DATE, temp_min_c, temp_max_c non nuls :", len(df_clean_essential))
print("Lignes sans AUCUN NaN :", len(df_all_complete))

# Garde df_clean_essential pour ne pas perdre trop de lignes à cause des variables optionnelles
df_final = df_clean_essential

# 5. Contrôle doublons, ordre des colonnes, tri et export
print("Doublons sur (arrondissement, DATE) :", df_final.duplicated(subset=["arrondissement", "DATE"]).sum())

cols = list(df_final.columns)
ordered_cols = ["arrondissement"] + (["DATE"] if "DATE" in cols else []) + [
    c for c in cols if c not in ["arrondissement", "DATE"]
]
df_final = df_final[ordered_cols].sort_values(["DATE", "arrondissement"]).reset_index(drop=True)

output_path = HERE / "df_final.csv"
df_final.to_csv(output_path, index=False)

print("Export terminé :", output_path, "| Shape :", df_final.shape)
df_final.head(5)

Top colonnes avec valeurs manquantes (%) :
wind_mean_ms                     78.22
frost_duration_min               67.13
hour_temp_max                    64.61
hour_temp_min                    64.60
temp_mean_c                      64.25
main_water_class                 38.42
water_total_area                 38.42
water_count                      38.42
temp_amplitude_c                 36.50
temp_mean_min_max_c              36.50
temp_min_c                       36.15
temp_max_c                       36.10
roads_avg_width_m_avenue         11.57
roads_total_surface_m2_avenue    11.57
roads_total_length_m_avenue      11.57
dtype: float64

Lignes avant filtre : 88208
Lignes avec DATE, temp_min_c, temp_max_c non nuls : 56013
Lignes sans AUCUN NaN : 11424
Doublons sur (arrondissement, DATE) : 0
Export terminé : df_final.csv | Shape : (56013, 43)


,arrondissement,DATE,green_count,green_total_area,green_calc_area,green_hort_area,green_perimeter,main_green_category,green_area_share_pct,arr_area_m2,...,roads_total_surface_m2_avenue,roads_total_surface_m2_boulevard,roads_total_surface_m2_rue,water_count,water_total_area,main_water_class,dni_wm2,wind_speed_10m_ms,wind_dir_10m_deg,pressure_pa
0,1,1990-05-01,16,50490.0,6731.0,19371.0,947.038315,Jardiniere,0.73,1830000,...,39280.0,56086.00,430080.95,3.0,166300.718992,Emprise voie d'eau : bief,183.455833,2.215833,323.208333,101623.75
1,3,1990-05-01,41,28612.0,28477.0,15489.0,2767.084225,Murs vegetalises,0.41,1170000,...,NaN,64144.50,283466.39,1.0,75.941599,Emprise voie d'eau : bief,183.455833,2.215833,323.208333,101623.75
2,4,1990-05-01,63,72445.0,69480.0,34099.0,10273.509163,Jardiniere,1.04,1600000,...,NaN,48088.75,186609.92,13.0,239021.597553,Emprise voie d'eau : embranchement,183.455833,2.215833,323.208333,101623.75
3,5,1990-05-01,96,93057.0,72374.0,37881.0,10211.666111,Jardiniere,1.34,2540000,...,97327.0,271085.00,441456.80,2.0,433.794234,Emprise voie d'eau : embranchement,183.455833,2.215833,323.208333,101623.75
4,6,1990-05-01,41,31810.0,31389.0,12077.0,3412.889510,Jardiniere,0.46,2150000,...,538.2,134748.00,446949.78,NaN,NaN,NaN,183.455833,2.215833,323.208333,101623.75
